In [1]:
import xarray as xr
import pandas as pd
import numpy as np


ds_sat_diff = xr.open_dataset('LGMR_data/ds_sat_diff.nc')
df_pre = pd.read_csv(r"D:\VScode\Inso_LGMR\inso_data\pre.csv")

ds_sat_diff.info()
df_pre.info()

xarray.Dataset {
dimensions:
	lat = 96 ;
	lon = 144 ;
	age = 119 ;

variables:
	float32 lat(lat) ;
		lat:FillValue = 9.969209968386869e+36 ;
		lat:long_name = vector latitudes ;
		lat:units = degrees_north ;
	float32 lon(lon) ;
		lon:FillValue = 9.969209968386869e+36 ;
		lon:long_name = vector of longitudes ;
		lon:units = degrees_east ;
	float32 age(age) ;
	float32 sat_diff(age, lat, lon) ;

// global attributes:
}<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     119 non-null    float64
 1   pre     119 non-null    float64
dtypes: float64(2)
memory usage: 2.0 KB


In [2]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy.spatial import distance
from scipy.stats import linregress
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

def interactive_ccm(ages, series1, series2, E=2, tau=1):
    """
    ages:       1D array of time points
    series1:    1D array (e.g. sat_diff) of same length as ages
    series2:    1D array (e.g. pre)    of same length as ages
    E, tau:     embedding dimension and delay
    """
    # 1) Build shadow manifolds & distance matrix
    def build_manifold(X):
        T, t0 = len(X), (E-1)*tau
        t_steps = np.arange(t0, T)
        M = np.zeros((T - t0, E))
        for i, t in enumerate(t_steps):
            for j in range(E):
                M[i,j] = X[t - j*tau]
        return M, t_steps

    M1, t_steps = build_manifold(series1)
    M2, _       = build_manifold(series2)
    dists       = distance.cdist(M1, M1)

    def get_neighbors(idx):
        return np.argsort(dists[idx])[1:E+2]

    # 2) Create a 4×2 grid, but merge right‐top (rows 1–2) and right‐bottom (3–4)
    specs = [
      [ {"rowspan":1}, {"rowspan":2} ],
      [ {"rowspan":1}, None           ],
      [ {"rowspan":1}, {"rowspan":2} ],
      [ {"rowspan":1}, None           ]
    ]
    titles = [
      "1) series1 TS",    "5) series2 & CCM",
      "2) series1 phase", "",
      "3) series2 TS",    "6) series2 vs pred",
      "4) series2 phase", ""
    ]

    fig = go.FigureWidget(make_subplots(
        rows=4, cols=2,
        column_widths=[0.4,0.6],
        specs=specs,
        subplot_titles=titles
    ))

    # for row in (2,4):   # the two phase‐space subplots
    #     fig.update_yaxes(
    #         scaleanchor = f"x{row*2-2}",  # matches the corresponding x‑axis
    #         scaleratio = 1,
    #         row=row, col=1
    #     )


    small_sz = 6
    # 3) Base (gray) traces on left
    left_panels = [
      (1, ages,       series1, 'lines+markers'),
      (2, M1[:,0],    M1[:,1], 'markers'),
      (3, ages,       series2, 'lines+markers'),
      (4, M2[:,0],    M2[:,1], 'markers'),
    ]
    for row, x, y, mode in left_panels:
        fig.add_trace(go.Scatter(
            x=x, y=y, mode=mode,
            marker=dict(color='lightgray', size=small_sz),
            showlegend=False
        ), row=row, col=1)

    # 4) Top‐right merged: raw series2 + CCM‐pred timeline
    #    (both go into row1,col2)
    fig.add_trace(go.Scatter(
        x=ages, y=series2, mode='lines+markers',
        marker=dict(color='lightgray', size=small_sz),
        name='series2 raw'
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[], y=[], mode='markers',
        marker=dict(color='black', size=8),
        name='CCM pred'
    ), row=1, col=2)

    # 5) Bottom‐right merged: scatter series2 vs predicted + fit line
    fig.add_trace(go.Scatter(
        x=[], y=[], mode='markers',
        marker=dict(size=8),
        name='series2 vs pred'
    ), row=3, col=2)
    fig.add_trace(go.Scatter(
        x=[], y=[], mode='lines', line=dict(dash='dash'),
        name='fit (ρ=–)'
    ), row=3, col=2)

    # 6) Add empty “highlight” traces on left for neighbors + target
    highlight_traces = []  # list of (nbr_idx, tgt_idx) per row
    for row in (1,2,3,4):
        # neighbors
        nbr_idx = len(fig.data)
        fig.add_trace(go.Scatter(x=[], y=[],
                                 mode='markers',
                                 marker=dict(size=10),
                                 showlegend=False),
                      row=row, col=1)
        # target
        tgt_idx = len(fig.data)
        fig.add_trace(go.Scatter(x=[], y=[],
                                 mode='markers',
                                 marker=dict(color='red', size=14),
                                 showlegend=False),
                      row=row, col=1)
        highlight_traces.append((nbr_idx, tgt_idx))

    # layout
    fig.update_layout(
        height=1100, width=800,
        margin=dict(l=40, r=40, t=60, b=40),
        title="Interactive CCM"
    )
    fig.update_xaxes(matches="x1", row=1, col=2)

    # 7) Button callback
    current = {'i':0}
    pred_t, pred_y = [], []

    btn = widgets.Button(description="▶️ next CCM step")
    out = widgets.Output()

    def step(_):
        i = current['i']
        if i >= len(t_steps):
            with out: print("done")
            return

        t    = t_steps[i]
        nbrs = get_neighbors(i)
        # weights & predict
        dvec    = dists[i,nbrs]
        w       = np.exp(-dvec/np.max([1e-6, dvec.min()]))
        w      /= w.sum()
        yhat    = (w * series2[t_steps[nbrs]]).sum()

        pred_t.append(t); pred_y.append(yhat)

        # update top‐right CCM pred trace (second trace in that cell → trace idx = 5)
        # base traces are 0–3 (left), 4=series2 raw, 5=CCM pred, 6=scatter,7=fit, then highlights
        fig.data[5].x = ages[pred_t]
        fig.data[5].y = pred_y

        # update bottom‐right scatter & fit
        # trace 6 = scatter, 7 = fit
        fig.data[6].x = series2[pred_t]
        fig.data[6].y = pred_y
        if len(pred_t) > 3:
            lr = linregress(series2[pred_t], pred_y)
            x0, x1 = series2[pred_t].min(), series2[pred_t].max()
            fig.data[7].x = [x0,x1]
            fig.data[7].y = [lr.intercept+lr.slope*x0,
                              lr.intercept+lr.slope*x1]
            fig.data[7].name = f"fit (ρ={lr.rvalue:.2f})"

        # # update highlights on left
        # for ridx, (nbr_idx, tgt_idx) in enumerate(highlight_traces):
        #     row = ridx+1
        #     # base geometry for that panel:
        #     if row in (1,3):
        #         xbase, ybase = (ages, series1) if row==1 else (ages, series2)
        #     else:
        #         m = M1 if row==2 else M2
        #         xbase, ybase = m[:,0], m[:,1]

        #     # neighbors
        #     xi = xbase[nbrs]
        #     yi = ybase[nbrs]
        #     colors = ['blue','green','orange','purple','brown','pink'][:len(nbrs)]
        #     fig.data[nbr_idx].update(x=xi, y=yi, marker_color=colors)

        #     # target
        #     xt, yt = (xbase[t], ybase[t])
        #     fig.data[tgt_idx].update(x=[xt], y=[yt])

        # current['i'] += 1

        # update highlights on left
        for ridx, (nbr_idx, tgt_idx) in enumerate(highlight_traces):
            row = ridx + 1

            if row == 1:
                xbase, ybase = ages, series1
                idx_use = t
            elif row == 2:
                xbase, ybase = M1[:,0], M1[:,1]
                idx_use = i                 # ← use manifold index
            elif row == 3:
                xbase, ybase = ages, series2
                idx_use = t
            else:  # row == 4
                xbase, ybase = M2[:,0], M2[:,1]
                idx_use = i                 # ← use manifold index

            # neighbors
            xi = xbase[nbrs]
            yi = ybase[nbrs]
            colors = ['blue','green','orange','purple','brown','pink'][:len(nbrs)]
            fig.data[nbr_idx].update(x=xi, y=yi, marker_color=colors)

            # target
            xt, yt = xbase[idx_use], ybase[idx_use]
            fig.data[tgt_idx].update(x=[xt], y=[yt])

        current['i'] += 1

    btn.on_click(step)
    display(btn, out)
    return fig




In [3]:
# --- usage: ---
ds     = xr.open_dataset('LGMR_data/ds_sat_diff.nc')
df_pre = pd.read_csv(r"D:\VScode\Inso_LGMR\inso_data\pre.csv")

ages = ds['age'].values
ts   = ds['sat_diff'].isel(lat=80, lon=0).values
pre  = df_pre['pre'].values

# call the function
fig = interactive_ccm(ages, ts, pre, E=2, tau=1)
fig

Button(description='▶️ next CCM step', style=ButtonStyle())

Output()

FigureWidget({
    'data': [{'marker': {'color': 'lightgray', 'size': 6},
              'mode': 'lines+markers',
              'showlegend': False,
              'type': 'scatter',
              'uid': '0fb8c8b4-0d35-48fe-9ec0-dca5c9496345',
              'x': array([  300.,   500.,   700.,   900.,  1100.,  1300.,  1500.,  1700.,  1900.,
                           2100.,  2300.,  2500.,  2700.,  2900.,  3100.,  3300.,  3500.,  3700.,
                           3900.,  4100.,  4300.,  4500.,  4700.,  4900.,  5100.,  5300.,  5500.,
                           5700.,  5900.,  6100.,  6300.,  6500.,  6700.,  6900.,  7100.,  7300.,
                           7500.,  7700.,  7900.,  8100.,  8300.,  8500.,  8700.,  8900.,  9100.,
                           9300.,  9500.,  9700.,  9900., 10100., 10300., 10500., 10700., 10900.,
                          11100., 11300., 11500., 11700., 11900., 12100., 12300., 12500., 12700.,
                          12900., 13100., 13300., 13500., 13700., 13900.

In [4]:

r_x = 3.8        
r_y = 3.5       
B_yx = 0.1      
LAG = 0          
t_total = 200    

X = np.zeros(t_total + 1)
Y = np.zeros(t_total + 1)
X[0] = 0.4
Y[0] = 0.2

for t in range(t_total):
    X[t+1] = r_x * X[t] * (1 - X[t])
    if (t + 1) < LAG:
        Y[t+1] = r_y * Y[t] * (1 - Y[t])
    else:
        Y[t+1] = r_y * Y[t] * (1 - Y[t]) + B_yx * X[t - LAG + 1]

time = np.arange(t_total + 1)
# plt.figure(figsize=(10, 4))
# plt.plot(time, X, 'b-', linewidth=2, label='X')
# plt.plot(time, Y, 'r-', linewidth=2, label='Y')
# plt.xlabel('Time')
# plt.ylabel('Value')
# plt.title(f'Simulated Time Series (X causes Y with lag = {LAG})')
# plt.legend()
# plt.show()


# save X and Y to csv and add a time column
df = pd.DataFrame({
    "Time": time,
    "X": X,
    "Y": Y,
})



ages = df['Time'].values
ts   = df['Y'].values
pre  = df['X'].values

# call the function
fig = interactive_ccm(ages, ts, pre, E=2, tau=1)
fig


Button(description='▶️ next CCM step', style=ButtonStyle())

Output()

FigureWidget({
    'data': [{'marker': {'color': 'lightgray', 'size': 6},
              'mode': 'lines+markers',
              'showlegend': False,
              'type': 'scatter',
              'uid': 'e502078a-23d4-4b79-8fbe-fd9990645706',
              'x': array([  0,   1,   2, ..., 198, 199, 200]),
              'xaxis': 'x',
              'y': array([0.2       , 0.6512    , 0.82548224, ..., 0.93197639, 0.31209879,
                          0.78498163]),
              'yaxis': 'y'},
             {'marker': {'color': 'lightgray', 'size': 6},
              'mode': 'markers',
              'showlegend': False,
              'type': 'scatter',
              'uid': '6ebbffc8-3c9f-4cda-aa8e-9c3bd123b300',
              'x': array([0.6512    , 0.82548224, 0.58476106, 0.90939717, 0.37991844, 0.85396101,
                          0.51541115, 0.93738562, 0.29379031, 0.76524923, 0.71921674, 0.73957679,
                          0.75783276, 0.69411434, 0.83799742, 0.49361627, 0.93206604, 0.31

In [32]:
import xarray as xr
import pandas as pd
import numpy as np
from scipy.spatial import distance
from scipy.stats import linregress
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# --- 1. Load data ---
ds     = xr.open_dataset('LGMR_data/ds_sat_diff.nc')
df_pre = pd.read_csv(r"D:\VScode\Inso_LGMR\inso_data\pre.csv")

lat_idx, lon_idx = 80, 0
E, tau           = 2, 1
ages             = ds['age'].values
ts               = ds['sat_diff'].isel(lat=lat_idx, lon=lon_idx).values
pre              = df_pre['pre'].values

# --- 2. Build shadow manifolds & distance matrix ---
def build_manifold(X, E, tau):
    T   = len(X)
    t0  = (E-1)*tau
    t_steps = np.arange(t0, T)
    M   = np.zeros((T - t0, E))
    for i, t in enumerate(t_steps):
        for j in range(E):
            M[i, j] = X[t - j*tau]
    return M, t_steps

M_ts, t_steps = build_manifold(ts, E, tau)
M_pre, _      = build_manifold(pre, E, tau)
dists         = distance.cdist(M_ts, M_ts)

def get_neighbors(idx):
    return np.argsort(dists[idx])[1:E+2]

# --- 3. Build the FigureWidget ---
fig = go.FigureWidget(make_subplots(
    rows=4, cols=2,
    column_widths=[0.4, 0.6],
    subplot_titles=[
      "1) sat_diff TS",         "5) pre TS (rhs)",
      "2) sat_diff phase‑space","6) CCM pred vs true",
      "3) pre TS",              "",
      "4) pre phase‑space",     ""
    ]
))

small_sz = 6

# Left column:
for (row, x, y) in [
    (1, ages, ts),
    (2, M_ts[:,0], M_ts[:,1]),
    (3, ages, pre),
    (4, M_pre[:,0], M_pre[:,1])
]:
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='lines+markers' if row in (1,3) else 'markers',
        marker=dict(color='lightgray', size=small_sz)
    ), row=row, col=1)

# Right column:
# row1 col2: duplicate pre TS
fig.add_trace(go.Scatter(x=ages, y=pre,
                         mode='lines+markers',
                         marker=dict(color='lightgray', size=small_sz),
                         showlegend=False),
              row=1, col=2)

# row2 col2: CCM predicted TS
fig.add_trace(go.Scatter(x=[], y=[],
                         mode='markers',
                         marker=dict(color='black', size=8),
                         name='CCM pred'),
              row=2, col=2)

# row4 col2: pre vs pred scatter + fit line
fig.add_trace(go.Scatter(x=[], y=[],
                         mode='markers',
                         marker=dict(size=8),
                         name='pre vs pred'),
              row=3, col=2)
fig.add_trace(go.Scatter(x=[], y=[],
                         mode='lines',
                         line=dict(dash='dash'),
                         name='fit (ρ=–)'),
              row=3, col=2)

fig.update_layout(
    height=1100, width=800,
    margin=dict(l=40, r=40, t=60, b=40),
    title_text="Interactive CCM"
)
fig.update_xaxes(matches="x1", row=2, col=2)

# --- 4. Interaction callback ---
current_step = {'i':0}
pred_times   = []
pred_yhats   = []

btn = widgets.Button(description="Predict → next")
out = widgets.Output()

def on_click(_):
    i = current_step['i']
    if i >= len(t_steps):
        with out: print("Finished.")
        return

    t    = t_steps[i]
    nbrs = get_neighbors(i)

    # calculate CCM prediction
    dist_vals = dists[i, nbrs]
    u = np.exp(-dist_vals/np.max([1e-6, dist_vals.min()]))
    w = u / u.sum()
    times_nbr = t_steps[nbrs]
    yhat      = (w * pre[times_nbr]).sum()

    pred_times.append(t)
    pred_yhats.append(yhat)

    # 1) Reset left-column traces
    for tr in [0,1,2,3]:
        n_pts = len(fig.data[tr].x)
        fig.data[tr].marker.color = ['lightgray'] * n_pts
        fig.data[tr].marker.size  = [small_sz] * n_pts

    # 2) Highlight the target in red & large
    highlight = dict(color='red', size=14)
    for tr, arr, idx in zip(
        [0,1,2,3],
        [ts, M_ts[:,0], pre, M_pre[:,0]],
        [t,   i,        t,   i]
    ):
        n_arr = len(arr)
        cols  = ['lightgray'] * n_arr
        szs   = [small_sz] * n_arr
        cols[idx] = highlight['color']
        szs[idx]  = highlight['size']
        fig.data[tr].marker.color = cols
        fig.data[tr].marker.size  = szs

    # 3) Color neighbors in distinct colors & medium size
    palette = ['blue','green','orange','purple','brown','pink']
    for k, nbr in enumerate(nbrs):
        c = palette[k]
        for tr, arr, map_idx in zip(
            [0,1,2,3],
            [ts, M_ts[:,0], pre, M_pre[:,0]],
            [nbr,    nbr,       times_nbr[k], nbr]
        ):
            cols = list(fig.data[tr].marker.color)
            szs  = list(fig.data[tr].marker.size)
            cols[map_idx] = c
            szs[map_idx]  = 10
            fig.data[tr].marker.color = cols
            fig.data[tr].marker.size  = szs

    # 4) Update CCM predicted TS (trace 5)
    fig.data[5].x = ages[pred_times]
    fig.data[5].y = pred_yhats
    fig.data[5].marker.color = ['black'] * len(pred_times)

    # 5) Update pre vs pred scatter (trace 6)
    fig.data[6].x = pre[pred_times]
    fig.data[6].y = pred_yhats

    # 6) Fit line after 3+ points (trace 7)
    if len(pred_times) > 3:
        x0, x1 = min(pre[pred_times]), max(pre[pred_times])
        lr = linregress(pre[pred_times], pred_yhats)
        fig.data[7].x = [x0, x1]
        fig.data[7].y = [lr.intercept + lr.slope * x0,
                         lr.intercept + lr.slope * x1]
        fig.data[7].name = f"fit (ρ={lr.rvalue:.2f})"
    else:
        fig.data[7].x = []
        fig.data[7].y = []
        fig.data[7].name = "fit (ρ=–)"

    current_step['i'] += 1

btn.on_click(on_click)
display(btn, out)
fig


Button(description='Predict → next', style=ButtonStyle())

Output()

FigureWidget({
    'data': [{'marker': {'color': 'lightgray', 'size': 6},
              'mode': 'lines+markers',
              'type': 'scatter',
              'uid': '4d42785f-c376-4c27-a293-1e90db4a7be1',
              'x': array([  300.,   500.,   700.,   900.,  1100.,  1300.,  1500.,  1700.,  1900.,
                           2100.,  2300.,  2500.,  2700.,  2900.,  3100.,  3300.,  3500.,  3700.,
                           3900.,  4100.,  4300.,  4500.,  4700.,  4900.,  5100.,  5300.,  5500.,
                           5700.,  5900.,  6100.,  6300.,  6500.,  6700.,  6900.,  7100.,  7300.,
                           7500.,  7700.,  7900.,  8100.,  8300.,  8500.,  8700.,  8900.,  9100.,
                           9300.,  9500.,  9700.,  9900., 10100., 10300., 10500., 10700., 10900.,
                          11100., 11300., 11500., 11700., 11900., 12100., 12300., 12500., 12700.,
                          12900., 13100., 13300., 13500., 13700., 13900., 14100., 14300., 14500.,
         